# Our own DeBERTa member — the one training this budget can finish

Every DeBERTa we can attach is frozen and worth about **0.523** balanced accuracy on unseen prompts (our own
audit). The 0.665 submission log replaced theirs with one fine-tuned on the clean pool (**0.5763**) and gained
on the leaderboard, and that pool is public.

The difference between this and the 8B LoRA is cost: **435M parameters over ~228k passages is two GPU-hours**
on 2×T4, where an 8B LoRA is 211 hours per epoch and therefore impossible on this quota.

We train in exactly the format the inference notebook already loads — `encoder/`, `head.safetensors`,
`meta.json` with the label order — so the member drops in through the existing loader with **no new code path**.

The loss is class-weighted by **sqrt inverse frequency**: the metric is balanced accuracy, and plain
cross-entropy on a pool whose classes run 20,579 to 6,594 would trade rare-class recall away. The holdout is
**prompt-disjoint** and drawn only from the official 2,699 rows, so it measures unseen prompts the way the
hidden test does.

In [ ]:
"""Train our own DeBERTa member — the one training this budget can actually finish.

Every public DeBERTa we can attach is frozen and worth about 0.523 balanced accuracy on unseen prompts.
The 0.665 submission log replaced it with one fine-tuned on the clean pool (0.5763) and gained on the LB,
and the pool is public. The difference between that and the 8B LoRA is cost: 435M parameters over ~228k
passages is two GPU-hours on 2xT4, where an 8B LoRA is 211 hours per epoch and therefore impossible here.

We train in exactly the format the inference notebook already loads — `encoder/`, `head.safetensors`,
`meta.json` with the label order — so the member drops in with the existing loader and no new code path.

Metric note: the competition scores balanced accuracy, so the loss is class-weighted by inverse frequency;
plain cross-entropy on a pool where the classes run 20,579 to 6,594 would trade rare-class recall away.
"""
import glob
import json
import os
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

T0 = time.time()
INPUT_DIR = "/kaggle/input"
OUT = "/kaggle/working/deberta_ours"
os.makedirs(OUT, exist_ok=True)

CFG = {
    "epochs": 2,
    "bs": 16,
    "max_len": 256,
    "lr_encoder": 1e-5,
    "lr_head": 1e-4,
    "warmup": 0.06,
    "val_frac": 0.15,          # prompt-grouped holdout from the official 2,699 rows
    "pool_cap_per_class": 6000,  # caps the biggest class so the epoch fits the budget
    "seed": 7,
}
print("CONFIG:", json.dumps(CFG), flush=True)


def find(pat):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pat}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"attach {pat}")
    return hits[0]


DATA = next(p for p in [f"{INPUT_DIR}/competitions/name-that-llm", f"{INPUT_DIR}/name-that-llm"]
            if os.path.exists(f"{p}/train.csv"))
train = pd.read_csv(f"{DATA}/train.csv")
LABELS = sorted(train.model.unique().tolist())
K = len(LABELS)
print(f"official train {len(train):,} rows, {train.prompt_id.nunique()} prompts, {K} labels", flush=True)

## Pool rows, capped and deduplicated; prompt-grouped holdout

In [ ]:
"""Pool rows: the merged corpus plus the api-gen and google-gen sets we already attach."""
frames = []
for pat, txtcol in (("ntl_merged*.parquet", "text"), ("merged*.parquet", "text"),
                    ("*api_gen*.parquet", "text"), ("*google_gen*.parquet", "text"),
                    ("pool*.parquet", "text")):
    for p in sorted(glob.glob(f"{INPUT_DIR}/**/{pat}", recursive=True)):
        try:
            d = pd.read_parquet(p)
        except Exception as e:
            print("skip", p, type(e).__name__, flush=True)
            continue
        cols = {c.lower(): c for c in d.columns}
        t = cols.get("text") or cols.get(txtcol)
        m = cols.get("model") or cols.get("label")
        if t is None or m is None:
            continue
        d = d[[t, m]].rename(columns={t: "text", m: "model"})
        frames.append(d)
        print(f"  {p.split('/')[-1]:<44} {len(d):>9,} rows", flush=True)

pool = pd.concat(frames, ignore_index=True)
pool = pool[pool.model.isin(LABELS)]
pool["text"] = pool.text.astype(str)
pool = pool[pool.text.str.split().str.len().between(30, 300)]
pool = pool.drop_duplicates(subset=["text"])
print(f"pool after filter/dedup: {len(pool):,} rows", flush=True)

rng = np.random.default_rng(CFG["seed"])
capped = [g.sample(min(len(g), CFG["pool_cap_per_class"]), random_state=CFG["seed"])
          for _, g in pool.groupby("model")]
pool = pd.concat(capped, ignore_index=True).sample(frac=1.0, random_state=CFG["seed"])
print(f"pool used: {len(pool):,} rows, per class "
      f"{pool.model.value_counts().min():,}-{pool.model.value_counts().max():,}", flush=True)

# prompt-grouped holdout from the OFFICIAL rows only: the pool has no competition prompts, so training on
# it and validating on the official holdout is a genuine unseen-prompt measurement.
prompts = np.array(sorted(train.prompt_id.unique()))
rng.shuffle(prompts)
n_val = max(1, int(len(prompts) * CFG["val_frac"]))
val_p = set(prompts[:n_val].tolist())
va = train[train.prompt_id.isin(val_p)].reset_index(drop=True)
tr_official = train[~train.prompt_id.isin(val_p)].reset_index(drop=True)
print(f"val {len(va):,} rows / {va.prompt_id.nunique()} prompts | "
      f"official-train {len(tr_official):,} rows (folded into the pool)", flush=True)

Xtr = pd.concat([pool[["text", "model"]], tr_official[["text", "model"]]], ignore_index=True)
Xtr = Xtr.sample(frac=1.0, random_state=CFG["seed"]).reset_index(drop=True)
print(f"training rows: {len(Xtr):,}", flush=True)

## The same encoder + pooled head the inference cell loads

In [ ]:
"""Model: the same encoder + pooled-linear-head shape the inference notebook already loads."""
from transformers import AutoConfig, AutoModel, AutoTokenizer

CKPT = os.path.dirname(find("deberta_v3_large/meta.json"))
enc_dir = os.path.join(CKPT, "encoder")
print("encoder from", enc_dir, flush=True)
tok = AutoTokenizer.from_pretrained(enc_dir)
config = AutoConfig.from_pretrained(enc_dir)
config.hidden_dropout_prob = 0.1
enc = AutoModel.from_pretrained(enc_dir, config=config).to("cuda:0")
H = enc.config.hidden_size
head = nn.Linear(H, K).to("cuda:0")

n_par = sum(p.numel() for p in enc.parameters()) + sum(p.numel() for p in head.parameters())
print(f"trainable parameters: {n_par:,} ({n_par/1e6:.0f}M)", flush=True)

lab2i = {m: i for i, m in enumerate(LABELS)}
counts = Xtr.model.map(lab2i).value_counts().reindex(range(K), fill_value=0).to_numpy(np.float64)
w_cls = (counts.sum() / (K * np.clip(counts, 1, None))) ** 0.5      # sqrt-inverse frequency: softer than 1/n
w_cls = torch.tensor(w_cls / w_cls.mean(), dtype=torch.float32, device="cuda:0")
print("class weights (min/median/max):",
      f"{w_cls.min():.2f}/{w_cls.median():.2f}/{w_cls.max():.2f}", flush=True)

opt = torch.optim.AdamW([
    {"params": enc.parameters(), "lr": CFG["lr_encoder"]},
    {"params": head.parameters(), "lr": CFG["lr_head"]},
], weight_decay=0.01)
steps_per_epoch = int(np.ceil(len(Xtr) / CFG["bs"]))
total_steps = steps_per_epoch * CFG["epochs"]
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=[CFG["lr_encoder"], CFG["lr_head"]], total_steps=total_steps,
    pct_start=CFG["warmup"], anneal_strategy="linear")
print(f"steps/epoch {steps_per_epoch:,} | total {total_steps:,}", flush=True)


def encode(texts, bs=None):
    """ids only; padding happens per batch so a short batch does not pay for a long one."""
    out = []
    bs = bs or 4096
    for s in range(0, len(texts), bs):
        out.extend(tok(list(texts[s:s + bs]), truncation=True, max_length=CFG["max_len"])["input_ids"])
    return out


@torch.inference_mode()
def evaluate(df):
    enc.eval()
    ids = encode(df.text.tolist())
    preds = np.zeros(len(ids), np.int64)
    for s in range(0, len(ids), 64):
        bt = tok.pad({"input_ids": ids[s:s + 64]}, return_tensors="pt").to("cuda:0")
        m = bt["attention_mask"].unsqueeze(-1).float()
        h = enc(**bt).last_hidden_state
        pooled = (h * m).sum(1) / m.sum(1).clamp(min=1)
        preds[s:s + 64] = head(pooled).argmax(1).cpu().numpy()
    enc.train()
    true = df.model.map(lab2i).to_numpy()
    per = [(preds[true == c] == c).mean() for c in range(K) if (true == c).any()]
    return float(np.mean(per)), preds, true

## Train, measuring holdout balanced accuracy after every epoch

In [ ]:
"""Train, tracking balanced accuracy on the unseen-prompt holdout after every epoch."""
enc.train()
ids_all = encode(Xtr.text.tolist())
labels_all = Xtr.model.map(lab2i).to_numpy(np.int64)
print(f"tokenised {len(ids_all):,} rows in {time.time()-T0:.0f}s", flush=True)

best = {"ba": -1.0, "epoch": -1}
rng_t = np.random.default_rng(CFG["seed"] + 1)
for epoch in range(CFG["epochs"]):
    order = rng_t.permutation(len(ids_all))
    run_loss, seen = 0.0, 0
    t_ep = time.time()
    for s in range(0, len(order), CFG["bs"]):
        idx = order[s:s + CFG["bs"]]
        if len(idx) < 2:
            continue
        bt = tok.pad({"input_ids": [ids_all[i] for i in idx]}, return_tensors="pt").to("cuda:0")
        m = bt["attention_mask"].unsqueeze(-1).float()
        h = enc(**bt).last_hidden_state
        pooled = (h * m).sum(1) / m.sum(1).clamp(min=1)
        logits = head(pooled)
        yb = torch.from_numpy(labels_all[idx]).to("cuda:0")
        loss = F.cross_entropy(logits, yb, weight=w_cls, label_smoothing=0.05)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(enc.parameters()) + list(head.parameters()), 1.0)
        opt.step()
        sched.step()
        opt.zero_grad(set_to_none=True)
        run_loss += float(loss.detach()) * len(idx)
        seen += len(idx)
        if (s // CFG["bs"]) % 200 == 0:
            print(f"  ep{epoch+1} {seen:>7,}/{len(order):,} loss {run_loss/max(seen,1):.4f} "
                  f"({time.time()-t_ep:.0f}s)", flush=True)
    ba, _, _ = evaluate(va)
    print(f"EPOCH {epoch+1}: loss {run_loss/max(seen,1):.4f} | HOLDOUT BA {ba:.4f} "
          f"({time.time()-t_ep:.0f}s)", flush=True)
    if ba > best["ba"]:
        best = {"ba": ba, "epoch": epoch + 1}
        enc.save_pretrained(os.path.join(OUT, "encoder"))
        tok.save_pretrained(os.path.join(OUT, "encoder"))
        torch.save({"weight": head.weight.detach().cpu(), "bias": head.bias.detach().cpu()},
                   os.path.join(OUT, "head.pt"))
        print(f"  saved (best so far, BA {ba:.4f})", flush=True)

print(f"\nBEST HOLDOUT BA {best['ba']:.4f} at epoch {best['epoch']}", flush=True)

## Write the drop-in checkpoint

In [ ]:
"""Write the checkpoint in the exact shape the inference notebook loads, and record the measurement."""
from safetensors.torch import save_file

ck = torch.load(os.path.join(OUT, "head.pt"), map_location="cpu")
save_file({"weight": ck["weight"].float(), "bias": ck["bias"].float()},
          os.path.join(OUT, "head.safetensors"))
os.remove(os.path.join(OUT, "head.pt"))

meta = {
    "labels": LABELS,                       # the inference cell asserts this equals its own LABELS
    "max_len": CFG["max_len"],
    "hidden_size": H,
    "pooling": "attention-masked mean of last_hidden_state",
    "trained_on": {
        "pool_rows": int(len(pool)),
        "official_rows": int(len(tr_official)),
        "total": int(len(Xtr)),
        "val_frac": CFG["val_frac"],
        "epochs": CFG["epochs"],
        "lr_encoder": CFG["lr_encoder"],
        "lr_head": CFG["lr_head"],
        "class_weight": "sqrt inverse frequency",
        "label_smoothing": 0.05,
        "seed": CFG["seed"],
    },
    "holdout_ba": round(float(best["ba"]), 4),
    "holdout_epoch": int(best["epoch"]),
    "holdout_prompts": int(va.prompt_id.nunique()),
    "holdout_rows": int(len(va)),
    "baseline_reference": {
        "note": "public frozen deberta measured 0.5230 unseen-prompt BA in our own audit",
        "published_pool_deberta": 0.5763,
    },
    "trained_in_seconds": round(time.time() - T0),
}
json.dump(meta, open(os.path.join(OUT, "meta.json"), "w"), indent=2)

print("=== deb_ours ===", flush=True)
print(json.dumps({k: meta[k] for k in ("holdout_ba", "holdout_epoch", "holdout_prompts", "trained_in_seconds")},
                 indent=1), flush=True)
print("files:", sorted(os.listdir(OUT)), flush=True)
print(f"encoder size {sum(os.path.getsize(os.path.join(OUT,'encoder',f)) for f in os.listdir(os.path.join(OUT,'encoder')))/2**20:.0f} MB",
      flush=True)
print("DROP-IN READY: attach this dataset and the existing deberta cell loads it unchanged", flush=True)